# Aligned green-up timing experiments

Compare timing rules on the same local imagery. The development comparison includes completed development batches registered since the saved preprocessing run.

1. **Strict:** The current checker requires the onset to be in the same month across the NDVI thresholds.
2. **Month boundary:** Try both starts when the main estimate itself straddles adjacent months.
3. **Threshold disagreement:** Also try both starts when the main estimate is within one month but the thresholds put the same growth cycle in two adjacent months. The other timing checks still apply.
4. **25-day onset gap:** Allow up to 25 days between the two observations that bracket green-up. The gap limit between later growth confirmations stays the same. Report the additional matched fields for development and primary JECAM.

Run the next cell with the project Python kernel. It needs a completed preprocessing feature run and its saved quality and window files. Allow several minutes for a large archive. The notebook reads the saved windows once and audits new batches locally. It can also run while other collection batches are pending.

The report counts distinct development fields across training and validation. A matched field must have at least one shared pixel location in the fixed and aligned windows after the overlap checks. Each candidate still has to pass the 80% imagery validity threshold and produce finite features in all eight bins. When both starts pass, the earlier one wins. The report includes fields gained and lost because a different start can change the matching pixels. The JECAM table is a coverage check on the historical benchmark.


In [ ]:
from pathlib import Path

from crop_pipeline.aligned_month_check import compare
from crop_pipeline.notebook_display import show_table

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
run, batches, new_groups, boundary, threshold, dev_gap, jecam_gap = compare(root)
print("Saved preprocessing run:", run.name)
print("Completed batches added:", ", ".join(batches) or "none")
print("Eligible fields from added batches:", len(new_groups))
show_table(boundary, title="Strict versus month-boundary coverage by crop")
show_table(
    [row for row in threshold if row["cohort"] == "matched"],
    title="Additional matched coverage from allowing threshold disagreement",
)
for population, rows in (("Development", dev_gap), ("Primary JECAM", jecam_gap)):
    matched = [row for row in rows if row["cohort"] == "matched"]
    if population == "Primary JECAM":
        matched = [
            {key: value for key, value in row.items() if not key.startswith("new_download_fields_")}
            for row in matched
        ]
    show_table(matched, title=f"{population} matched fields with a 25-day onset gap")
